In [6]:
import pandas as pd
import numpy as np
import pandera as pa
from pandera import Column, Check, DataFrameSchema
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/cleaned/ipo_cleaned.csv")
df["Date"] = pd.to_datetime(df["Date"])

print(f"Shape: {df.shape}")
print("Data loaded!")

Shape: (259, 20)
Data loaded!


In [7]:
# This is like a contract — we define exactly what
# our data MUST look like before feeding it to ML models

schema = DataFrameSchema({
    "ipo_name": Column(str, nullable=False),

    "issue_size_cr": Column(float,
        Check.greater_than(0),
        nullable=False
    ),

    "issue_price": Column(float,
        Check.greater_than(0),
        nullable=False
    ),

    "total_subscription": Column(float,
        Check.greater_than_or_equal_to(0),
        nullable=False
    ),

    "listing_gains_pct": Column(float,
        Check.in_range(-100, 1000),
        nullable=False
    ),

    "performance_category": Column(str,
        Check.isin(["Loss", "Weak", "Moderate",
                    "Strong", "Blockbuster"]),
        nullable=False
    ),

    "Year": Column(int,
        Check.in_range(2010, 2021),
        nullable=False
    ),

    "Month": Column(int,
        Check.in_range(1, 12),
        nullable=False
    ),

    "qib_subscription": Column(float,
        Check.greater_than_or_equal_to(0),
        nullable=False
    ),

    "hni_subscription": Column(float,
        Check.greater_than_or_equal_to(0),
        nullable=False
    ),

    "rii_subscription": Column(float,
        Check.greater_than_or_equal_to(0),
        nullable=False
    ),
})

print("Schema defined successfully!")

Schema defined successfully!


In [8]:
try:
    validated_df = schema.validate(df, lazy=True)
    print("✓ All validation checks passed!")
    print(f"Validated shape: {validated_df.shape}")

except pa.errors.SchemaErrors as err:
    print("✗ Validation failed! Here are the issues:")
    print(err.failure_cases)

✓ All validation checks passed!
Validated shape: (259, 20)


In [9]:
print("=== CUSTOM DATA QUALITY CHECKS ===\n")

# Check 1: No duplicate IPO names
dupes = df["ipo_name"].duplicated().sum()
print(f"Duplicate IPO names: {dupes} "
      f"{'✓' if dupes == 0 else '✗ Fix needed'}")

# Check 2: Listing close should be > 0
invalid_close = (df["listing_close"] <= 0).sum()
print(f"Invalid listing close prices: {invalid_close} "
      f"{'✓' if invalid_close == 0 else '✗ Fix needed'}")

# Check 3: listing_gains_pct should match listing_close and issue_price
df["calculated_gain"] = (
    (df["listing_close"] - df["issue_price"]) / df["issue_price"]
) * 100

diff = (df["listing_gains_pct"] - df["calculated_gain"]).abs()
mismatch = (diff > 1.0).sum()
print(f"Gain calculation mismatches (>1%): {mismatch} "
      f"{'✓' if mismatch == 0 else '! Minor rounding differences'}")

# Check 4: Date range is correct
min_date = df["Date"].min()
max_date = df["Date"].max()
print(f"Date range: {min_date.date()} to {max_date.date()} ✓")

# Check 5: No future dates
future = (df["Date"] > pd.Timestamp.today()).sum()
print(f"Future dates found: {future} "
      f"{'✓' if future == 0 else '✗ Fix needed'}")

print("\nAll custom checks complete!")

=== CUSTOM DATA QUALITY CHECKS ===

Duplicate IPO names: 0 ✓
Invalid listing close prices: 0 ✓
Gain calculation mismatches (>1%): 0 ✓
Date range: 2010-02-03 to 2021-07-29 ✓
Future dates found: 0 ✓

All custom checks complete!


In [10]:
import os
os.makedirs("../data/validated", exist_ok=True)

# Drop the temporary calculated column
df = df.drop(columns=["calculated_gain"])

df.to_csv("../data/validated/ipo_validated.csv", index=False)

print("Validated data saved to data/validated/ipo_validated.csv")
print(f"Final shape: {df.shape}")
print("\nData pipeline complete:")
print("  Raw → Cleaned → Validated ✓")

Validated data saved to data/validated/ipo_validated.csv
Final shape: (259, 20)

Data pipeline complete:
  Raw → Cleaned → Validated ✓
